# conv-padding-zero — worked example 3: Zero-pad 2-D input so output size is preserved under a 3x3 window

> Worked example from [Delta Drills](https://delta-drills.vercel.app). Atom: `conv-padding-zero`.

**This is a worked example — read it, run it, follow the reasoning.** It is study material, not a graded drill (no completion beacon). When the steps feel obvious, move to the faded version, then the full drill.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)

## Concept

'Same' padding chooses the pad width so that sliding a `K`-wide window with stride 1 produces an output the same spatial size as the input. For an odd kernel `K`, that pad is `p = (K - 1) // 2` on every side. We build the padded buffer manually and confirm the post-conv output size.

## Worked solution

For a `(B, IC, H, W)` input and a `K x K` stride-1 window, the valid-conv output height is `H_pad - K + 1` where `H_pad` is the padded height. To keep it equal to `H`, we need `H_pad = H + K - 1`, i.e. `p = (K - 1) // 2` on each side (valid because `K` is odd).

1. **Compute the pad.** `p = (K - 1) // 2`.
2. **Allocate the symmetric 2-D buffer.** `out = x.new_zeros(B, IC, H + 2*p, W + 2*p)` — zeros everywhere, dtype/device inherited.
3. **Assign the interior.** `out[..., p : p + H, p : p + W] = x` copies the image into the center, leaving a `p`-thick zero frame on all four sides.
4. **Verify the size identity.** After padding, `(H + 2*p) - K + 1 == H` because `2*p = K - 1`. We print this to make the 'same' property concrete.

In [ ]:
def pad2d_same(x: Tensor, K: int) -> Tensor:
    B, IC, H, W = x.shape
    p = (K - 1) // 2
    out = x.new_zeros(B, IC, H + 2 * p, W + 2 * p)
    out[..., p : p + H, p : p + W] = x
    return out

t.manual_seed(0)
x = t.randn(1, 2, 5, 7)
K = 3
y = pad2d_same(x, K)
out_h = y.shape[2] - K + 1
out_w = y.shape[3] - K + 1
print(tuple(y.shape))
print('preserves size:', out_h == x.shape[2] and out_w == x.shape[3])
print('frame zero:', bool((y[..., 0, :] == 0).all()) and bool((y[..., :, 0] == 0).all()))